In [8]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone

load_dotenv()  # loads .env from backend folder
api_key = os.getenv("PINECONE_API_KEY")
env = os.getenv("PINECONE_ENV")

pc = Pinecone(api_key=api_key, environment=env)
print("Connected to Pinecone environment:", env)
# Use the client object 'pc' for all operations
print("Indexes available:", pc.list_indexes())

Connected to Pinecone environment: us-east-1
Indexes available: [{
    "name": "ikarus-products",
    "metric": "cosine",
    "host": "ikarus-products-z7fr8o5.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null
}]


In [26]:
# Phase_2B_retrieval_test.ipynb / retrieval_test.py
# Requirements:
# pip install sentence-transformers numpy pandas pinecone-client==deprecated? (we'll try 'pinecone' first)
# Prefer: pip install sentence-transformers pinecone

import os
import sys
import time
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# ---- CONFIG ----
# API_KEY = os.getenv("PINECONE_API_KEY")
env = os.getenv("PINECONE_ENV")   # e.g. "asia-southeast1-gcp"
INDEX_NAME = os.getenv("PINECONE_INDEX")
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "__default__")  # set if you used a specific namespace

TOP_K = 10

# NEW: Add the host from the output of Cell 1
INDEX_HOST = "https://ikarus-products-z7fr8o5.svc.aped-4627-b74a.pinecone.io"

In [27]:
# Quick sanity checks
if api_key.startswith("<") or env.startswith("<"):
    print("⚠️  Fill in PINECONE_API_KEY and PINECONE_ENV in env variables or update the script.")
    # Continue anyway so user can see the code

print("Loading sentence-transformers model (all-mpnet-base-v2) ...")
model = SentenceTransformer("all-mpnet-base-v2")  # same model used to create index

Loading sentence-transformers model (all-mpnet-base-v2) ...


In [28]:

# ---- Query function (works with both clients) ----
def embed_text(texts):
    """Return L2-normalized embeddings for a list of texts as numpy array (shape: n x 768)."""
    # Assuming 'model' is defined and initialized correctly in another cell
    arr = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    # unit-normalize (safe even if already normalized)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    arr = arr / norms
    return arr

def query_pinecone_vector(vector: np.ndarray, top_k=TOP_K, namespace=NAMESPACE):
    """Query the Pinecone index using the new-style Pinecone client object 'pc'."""
    # Ensure the vector is in list format for the query
    vect_list = vector.tolist() if isinstance(vector, np.ndarray) else vector
    
    # Use the 'pc' client instance (initialized successfully in Cell 1)
    try:
        # 1. Get the Index object from the Client object 'pc'
        # This is the CORRECT NEW CLIENT SYNTAX
        idx = pc.Index(host=INDEX_HOST)
        
        # 2. Call query on the Index object 'idx' (BYOV method)
        resp = idx.query(
            vector=vect_list, 
            top_k=top_k, 
            include_values=False, 
            include_metadata=True, 
            namespace=namespace
        )
        return resp
    except Exception as e:
        # Reraise the exception for diagnostics
        raise RuntimeError(f"Pinecone index query failed: {e}")
        
def parse_and_print_response(resp):
    """Parse common response shapes and print readable output."""
    # Two common shapes:
    # - new client: resp.matches (list), each has id, score, metadata
    # - legacy: resp.get("matches") or resp.matches
    matches = None
    if hasattr(resp, "matches"):
        matches = resp.matches
    elif isinstance(resp, dict) and "matches" in resp:
        matches = resp["matches"]
    elif isinstance(resp, dict) and "results" in resp:
        # some wrappers nest differently
        try:
            matches = resp["results"][0]["matches"]
        except Exception:
            matches = None

    if not matches:
        print("No matches returned or unable to parse response structure. Raw response:")
        print(json.dumps(resp, default=str, indent=2))
        return

    rows = []
    print(f"\nTop {len(matches)} results:")
    for i, m in enumerate(matches, start=1):
        # different clients use different field names ('score' vs 'similarity')
        score = m.get("score") if isinstance(m, dict) else getattr(m, "score", None)
        if score is None:
            score = m.get("similarity") if isinstance(m, dict) else getattr(m, "similarity", None)
        mid = m.get("id") if isinstance(m, dict) else getattr(m, "id", None)
        metadata = m.get("metadata") if isinstance(m, dict) else getattr(m, "metadata", None)
        title = metadata.get("title") if isinstance(metadata, dict) and "title" in metadata else metadata
        print(f"{i:02d}. id={mid}  score={score:.4f}  metadata keys={list(metadata.keys()) if isinstance(metadata, dict) else type(metadata)}")
        # print a short summary of metadata (title, brand, price)
        preview = {}
        if isinstance(metadata, dict):
            for k in ("title", "brand", "price_clean", "category_list"):
                if k in metadata:
                    preview[k] = metadata[k]
        print("    preview:", preview)
        rows.append({"rank": i, "id": mid, "score": score, **(preview or {})})
    return pd.DataFrame(rows)


In [29]:
# ---- Quick Test Prompts ----
test_prompts = [
    "modern wooden dining table for 6 people",
    "compact bedside table with drawer, oak finish",
    "scandinavian style lounge chair in light grey fabric",
    "outdoor patio wicker sofa set, weather resistant",
    "minimalist wooden coffee table with storage shelf"
]

In [30]:
# ---- Run tests ----
results = {}
for p in test_prompts:
    print("\n" + "="*80)
    print("Query prompt:", p)
    v = embed_text([p])[0]
    # Sanity: embedding dim
    print("embedding dim:", v.shape)
    try:
        resp = query_pinecone_vector(v, top_k=TOP_K, namespace=NAMESPACE)
    except Exception as e:
        print("❌ Pinecone query failed:", e)
        resp = None

    if resp is not None:
        df = parse_and_print_response(resp)
        results[p] = df
    else:
        results[p] = None
    time.sleep(0.15)

# ---- Diagnostics & heuristics ----
print("\n" + "="*80)
print("Diagnostics summary:")
for prompt, df in results.items():
    print("\nPrompt:", prompt)
    if df is None:
        print("  No result dataframe (query failure).")
        continue
    if df.empty:
        print("  Empty results.")
        continue
    top_score = float(df.loc[0, "score"])
    mean_top5 = float(df.loc[:4, "score"].mean()) if len(df) >= 5 else float(df["score"].mean())
    print(f"  top score = {top_score:.4f}, mean(top5) = {mean_top5:.4f}")

    # Heuristics for cosine (unit-normalized vectors -> dot product in [-1,1])
    if top_score > 0.8:
        verdict = "Excellent"
    elif top_score > 0.65:
        verdict = "Good"
    elif top_score > 0.45:
        verdict = "Marginal"
    else:
        verdict = "Poor"

    print("  retrieval quality (heuristic):", verdict)
    display(df.head(5))

print("\nDone. If results look good (Good/Excellent for most prompts), we're ready to add the FastAPI endpoint.")



Query prompt: modern wooden dining table for 6 people
embedding dim: (768,)

Top 10 results:
01. id=91688563-4731-5976-bc36-85e98ca7ba1a::chunk0  score=0.6453  metadata keys=['brand', 'category_list', 'chunk_count', 'chunk_index', 'combined_len', 'price', 'title', 'uniq_id']
    preview: {'title': 'yuihome extendable round, farmhouse 16" leaf table for dining room, kitchen,natural wood wash', 'brand': 'yuihome', 'category_list': "['home & kitchen', 'furniture', 'dining room furniture', 'tables']"}
02. id=950be137-d751-5d57-a37d-d6741f2c94a0  score=0.6050  metadata keys=['brand', 'category_list', 'chunk_count', 'chunk_index', 'combined_len', 'price', 'title', 'uniq_id']
    preview: {'title': 'vecelo modern industrial style 3-piece dining room kitchen table and pu cushion chair sets for small space, 2, retro brown', 'brand': 'vecelo store', 'category_list': "['home & kitchen', 'furniture', 'dining room furniture', 'table & chair sets']"}
03. id=ca08facd-e8da-5a55-84df-22c1aea973ea  sco

,rank,id,score,title,brand,category_list
0,1,91688563-4731-5976-bc36-85e98ca7ba1a::chunk0,0.645298,"yuihome extendable round, farmhouse 16"" leaf t...",yuihome,"['home & kitchen', 'furniture', 'dining room f..."
1,2,950be137-d751-5d57-a37d-d6741f2c94a0,0.604987,vecelo modern industrial style 3-piece dining ...,vecelo store,"['home & kitchen', 'furniture', 'dining room f..."
2,3,ca08facd-e8da-5a55-84df-22c1aea973ea,0.595065,"casual home 5 piece tray table set, espresso",casual home store,"['home & kitchen', 'furniture', 'living room f..."
3,4,0db5b842-14ad-5c09-a607-f138120e1e78::chunk0,0.593348,"lokkhan industrial bar table 38.6""-48.4"" heigh...",lokkhan store,"['home & kitchen', 'furniture', 'dining room f..."
4,5,0f7db59b-10e3-5fce-a0c4-380f718befe1,0.586093,"get set style black glass side table, square g...",get set style store,"['home & kitchen', 'furniture', 'living room f..."



Prompt: compact bedside table with drawer, oak finish
  top score = 0.6526, mean(top5) = 0.6179
  retrieval quality (heuristic): Good


,rank,id,score,title,brand,category_list
0,1,06e3b025-a397-5081-8078-9c2140875a89,0.652632,seventable nightstand with charging station an...,seventable store,"['home & kitchen', 'furniture', 'bedroom furni..."
1,2,ef88909a-9a2c-54ba-9cae-c2adef3df7ff,0.631228,new classic furniture evander wood end table w...,new classic furniture store,"['home & kitchen', 'furniture', 'living room f..."
2,3,9a6699ab-a3f6-5ef6-8630-74731962e81c,0.607131,"lizipai floating bedside table, no assembly re...",lizipai,"['home & kitchen', 'furniture', 'bedroom furni..."
3,4,07b9d03a-02bc-5bc9-9133-a8e7b706cbc4,0.603950,furniturer 27h round drawer 2 tiers endtable n...,furniturer,"['home & kitchen', 'furniture', 'living room f..."
4,5,d553e7ea-4dce-5147-8534-3230f5e7848d,0.594338,"domydevm black end table, nightstand with char...",domydevm store,"['home & kitchen', 'furniture', 'living room f..."



Prompt: scandinavian style lounge chair in light grey fabric
  top score = 0.6867, mean(top5) = 0.6141
  retrieval quality (heuristic): Good


...


Prompt: outdoor patio wicker sofa set, weather resistant
  top score = 0.4976, mean(top5) = 0.4638
  retrieval quality (heuristic): Marginal


,rank,id,score,title,brand,category_list
0,1,fe25ae1d-4a82-57ad-9bab-b9de4321fd0b,0.497585,karl home accent chair mid-century modern chai...,karl home store,"['home & kitchen', 'furniture', 'living room f..."
1,2,780dda1f-e7ef-598e-9c0e-6536c4b4c261,0.465276,stylish camping mings mark rc4 reversible clas...,stylish camping store,"['patio', 'lawn & garden', 'outdoor dcor', 'do..."
2,3,37323128-75a9-578b-9378-79653bfd5b52,0.461320,fanye oversized 6 seaters modular storage sect...,fanye,"['home & kitchen', 'furniture', 'living room f..."
3,4,d00f8e82-7c15-5c18-9835-21574ca361cf,0.451236,monibloom round folding faux fur saucer chair ...,monibloom store,"['home & kitchen', 'furniture', 'game & recrea..."
4,5,850a2c03-f708-59b4-a506-c4b9428bb2ea::chunk0,0.443348,"cordaroys chenille bean bag ottoman footstool,...",cordaroys store,"['home & kitchen', 'furniture']"



Prompt: minimalist wooden coffee table with storage shelf
  top score = 0.6240, mean(top5) = 0.5730
  retrieval quality (heuristic): Marginal


,rank,id,score,title,brand,category_list
0,1,487adf3a-9485-5500-9c98-bcc391eda169,0.624039,"3-tier side table,narrow end table with storag...",hometodou,"['home & kitchen', 'furniture', 'living room f..."
1,2,ef88909a-9a2c-54ba-9cae-c2adef3df7ff,0.570492,new classic furniture evander wood end table w...,new classic furniture store,"['home & kitchen', 'furniture', 'living room f..."
2,3,930327b1-8e0e-5771-a69c-8e1d34a09694::chunk0,0.562187,"plebs home solid desktop store cart, with rubb...",plebs home,"['home & kitchen', 'furniture', 'kitchen furni..."
3,4,e5596d53-58a9-528c-af7f-9ae10b5ceba2,0.556459,walker edison furniture modern round nesting c...,walker edison store,"['home & kitchen', 'furniture', 'living room f..."
4,5,2c7902b4-ae0c-5ed9-9cbc-b6927d0b3347::chunk1,0.551997,"furinno coffee table with bins, espressobrown ...",furinno store,"['home & kitchen', 'furniture', 'living room f..."



Done. If results look good (Good/Excellent for most prompts), we're ready to add the FastAPI endpoint.
